**Note**: needs to be run separetely in the rtv environment (not the MBM). Computes the sky view factor from DEMs (needed as inputs in the MBM).

In [ ]:
# --- Standard library ---
import os
import glob
import shutil
import warnings
import sys

sys.path.append(os.path.join(os.getcwd(), "../../"))

# --- Scientific / numeric ---
import numpy as np
import xarray as xr

# --- Geospatial ---
import rasterio
import rioxarray
import rvt.default
import rvt.vis

# --- Plotting / progress ---
import matplotlib.pyplot as plt
from tqdm import tqdm

from regions.TF_Europe.scripts.svf_processing import *
from regions.TF_Europe.scripts.config_TF_Europe import *

# --- Misc settings ---
warnings.filterwarnings("ignore")
cfg_datapath = '/scratch-3/vmarijn/MassBalanceMachine/../data/'

In [ ]:
# RVT parameters (tune as needed)
SVF_N_DIR = 16
SVF_R_MAX = 10
SVF_NOISE = 0
ASVF_LEVEL = 1
ASVF_DIR = 315

DST_CRS = "EPSG:4326"  # WGS84 lat/lon

# Variables we expect (only reproject those that exist)
DATA_VARS = ["svf", "asvf", "opns"]

In [ ]:
RGI_REGIONS

In [ ]:
import pandas as pd

# df_stakes = pd.read_csv(os.path.join(cfg_datapath, 'WGMS/raw/mass_balance_point.csv'))
# df_RGIId = pd.read_csv(cfg_datapath + 'WGMS/raw/glacier.csv')

# # Create mapping dictionaries from id to rgi60_ids and gtng_region
# id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))
# id_to_gtng_region_map = dict(zip(df_RGIId['id'], df_RGIId['gtng_region']))

# # Add RGIId and gtng_region columns to df_stakes
# df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)
# df_stakes['gtng_region'] = df_stakes['glacier_id'].map(id_to_gtng_region_map)

# # Save unique RGIIds per gtng_region
# rgi_per_region = df_stakes.dropna(subset=['RGIId', 'gtng_region']) \
#                            .groupby('gtng_region')['RGIId'] \
#                            .unique() \
#                            .to_dict()

# # Print summary
# for region, rgis in rgi_per_region.items():
#     print(f"{region}: {len(rgis)} glaciers")

df_stakes_US_CA = pd.read_csv(
    os.path.join(cfg_datapath, 'WGMS/raw/RGIids_stakes_US_CA.csv'))

# Save unique RGIIds per gtng_region
rgi_per_region = df_stakes_US_CA.dropna(subset=['RGIId', 'RGI_REGION']) \
                           .groupby('RGI_REGION')['RGIId'] \
                           .unique() \
                           .to_dict()

# Print summary
for region, rgis in rgi_per_region.items():
    print(f"{region}: {len(rgis)} glaciers")

## RGI:

### Compute skyview factor:

RGI_v6/
  ├── RGI_06_Iceland/
  │     ├── geotiff/
  │     ├── svf_nc_latlon/
  │
  ├── RGI_07_Svalbard/
  │     ├── geotiff/
  │     ├── svf_nc_latlon/
  │
  ...

In [ ]:
EMPTY_OUTPUT_FIRST = True
SAVE_XY = False


def process_region(cfg_datapath, rgi_region, spec, allowed_rgis=None):
    region_folder = spec["folder"]
    path_geotiff = os.path.join(cfg_datapath, "RGI_v6", region_folder, "geotiff")
    path_nc_ll = os.path.join(cfg_datapath, "RGI_v6", region_folder, "svf_nc_latlon")
    path_nc_xy = os.path.join(cfg_datapath, "RGI_v6", region_folder, "svf_nc_xy")

    if not os.path.isdir(path_geotiff):
        print(f"[{rgi_region}] No geotiff folder found, skipping.")
        return

    # clean outputs
    if EMPTY_OUTPUT_FIRST:
        if os.path.exists(path_nc_ll):
            shutil.rmtree(path_nc_ll)
        os.makedirs(path_nc_ll, exist_ok=True)
        if SAVE_XY:
            if os.path.exists(path_nc_xy):
                shutil.rmtree(path_nc_xy)
            os.makedirs(path_nc_xy, exist_ok=True)
    else:
        os.makedirs(path_nc_ll, exist_ok=True)
        if SAVE_XY:
            os.makedirs(path_nc_xy, exist_ok=True)

    tif_list = sorted(
        glob.glob(os.path.join(path_geotiff, f"RGI60-{rgi_region}.*.tif"))
    )

    # Filter to stake-only RGIs if an allowed list is provided
    if allowed_rgis is not None:
        tif_list = [
            p for p in tif_list
            if os.path.splitext(os.path.basename(p))[0] in allowed_rgis
        ]
        print(f"\n[{rgi_region} - {spec['name']}] Filtered to {len(tif_list)} stake DEMs.")
    else:
        print(f"\n[{rgi_region} - {spec['name']}] Found {len(tif_list)} DEMs.")

    for dem_path in tqdm(tif_list, desc=f"Region {rgi_region} SVF → lat/lon"):
        base = os.path.splitext(os.path.basename(dem_path))[0]
        out_nc_ll = os.path.join(path_nc_ll, f"{base}_svf_latlon.nc")
        out_nc_xy = os.path.join(path_nc_xy, f"{base}_svf_xy.nc")
        try:
            ds_xy = process_dem_to_netcdf(
                dem_path,
                svf_n_dir=SVF_N_DIR,
                svf_r_max=SVF_R_MAX,
                svf_noise=SVF_NOISE,
                asvf_level=ASVF_LEVEL,
                asvf_dir=ASVF_DIR,
            )
            ds_ll = reproject_ds_to_latlon(
                ds_xy,
                data_vars=DATA_VARS,
                dst_crs=DST_CRS,
            )
            encoding = {
                v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                for v in ds_ll.data_vars
            }
            ds_ll.to_netcdf(out_nc_ll, engine="h5netcdf", encoding=encoding)
            if SAVE_XY:
                enc_xy = {
                    v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                    for v in ds_xy.data_vars
                }
                ds_xy.to_netcdf(out_nc_xy, engine="h5netcdf", encoding=enc_xy)
        except Exception as e:
            print(f"[{rgi_region}] Failed on {base}: {e}")

# -------------------
# Run for all regions
# -------------------
# Mapping from rgi_region number to gtng_region key in rgi_per_region
# Regions that should be filtered to stake-only RGIs
STAKE_ONLY_REGIONS = {"01", "02"}  # Alaska, W. Canada & US, Greenland

RGI_TO_GTNG = {
    "01": "01_alaska",
    "02": "02_western_canada_usa",
}

for rgi_region, spec in RGI_REGIONS.items():
    if rgi_region in RGI_TO_GTNG:
        gtng_key = RGI_TO_GTNG[rgi_region]
        allowed_rgis = set(rgi_per_region.get(gtng_key, []))
        print(f"[{rgi_region}] Filtering to {len(allowed_rgis)} stake RGIs.")
    else:
        allowed_rgis = None
    process_region(cfg_datapath, rgi_region, spec, allowed_rgis=allowed_rgis)

In [ ]:
# Look at an example
# --- Path to one of your generated SVF NetCDFs ---
rgi_gl = "RGI60-07.00001"
path_nc_ll = os.path.join(cfg_datapath, "RGI_v6", RGI_REGIONS['07']['folder'], "svf_nc_latlon",)
path_svf = os.path.join(
    cfg_datapath,
    path_nc_ll,
    f"{rgi_gl}_svf_latlon.nc",
)

ds = xr.open_dataset(path_svf)

# --- Handle coordinate names flexibly ---
if "lon" in ds.coords and "lat" in ds.coords:
    xcoord, ycoord = "lon", "lat"
elif "x" in ds.coords and "y" in ds.coords:
    # rename for clarity if needed
    ds = ds.rename({"x": "lon", "y": "lat"})
    xcoord, ycoord = "lon", "lat"
else:
    raise ValueError("Could not find lon/lat coordinates in dataset")

# Bounds
lon_min, lon_max = float(ds[xcoord].min()), float(ds[xcoord].max())
lat_min, lat_max = float(ds[ycoord].min()), float(ds[ycoord].max())

print(f"lon: {lon_min:.5f} to {lon_max:.5f}")
print(f"lat: {lat_min:.5f} to {lat_max:.5f}")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

# SVF
ds["svf"].plot(ax=axes[0], x=xcoord, y=ycoord, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Sky-View Factor (SVF)")
axes[0].set_xlabel("Longitude (°)")
axes[0].set_ylabel("Latitude (°)")
axes[0].set_aspect("equal")

# ASVF
if "asvf" in ds:
    ds["asvf"].plot(ax=axes[1],
                    x=xcoord,
                    y=ycoord,
                    cmap="viridis",
                    vmin=0,
                    vmax=1)
    axes[1].set_title("Anisotropic SVF (ASVF)")
    axes[1].set_xlabel("Longitude (°)")
    axes[1].set_aspect("equal")
else:
    axes[1].set_visible(False)

# OPNS
if "opns" in ds:
    ds["opns"].plot(ax=axes[2], x=xcoord, y=ycoord, cmap="terrain")
    axes[2].set_title("Positive Openness (OPNS)")
    axes[2].set_xlabel("Longitude (°)")
    axes[2].set_aspect("equal")
else:
    axes[2].set_visible(False)

plt.suptitle(f"{rgi_gl} — SVF products (lat/lon)", fontsize=14)
plt.show()